# 통신망 방호: 문제 생성과 전략 실행 시연

**시연 목표:** 시드로 같은 문제를 생성하고, 문제 파일을 표준 입력으로 전략에 전달한 뒤, 출력된 방호안의 점수를 확인한다. 이 노트북은 중간평가 시연 초안이다.

**담당 계획:** 김대건 — 문제 생성과 차수 기준선. 진명호 — 공통 평가와 완전탐색 원리. 신동윤·김연우 — 각자 직접 작성하고 실행한 전략. 개인별 실제 작성 범위와 AI 사용 내역은 본인이 확인한다.

아래 기준선 예시는 기존 AI 연구 보조 코드다. 다른 팀원의 코드를 대신 작성한 것으로 표시하지 않는다.

## 1. 실행 준비

Linux에서는 저장소 루트 또는 `presentation` 폴더에서 실행한다. Colab에서는 저장소를 가져온다. 저장소가 비공개라면 접근 권한이 필요하다. Python의 NumPy, SciPy, NetworkX가 필요하다. 실행 중 생성하는 파일은 임시 폴더에 둔다.

In [1]:
from pathlib import Path
import os, sys, subprocess, tempfile

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

here = Path.cwd().resolve()
if (here / 'protection').is_dir():
    root = here
elif (here.parent / 'protection').is_dir():
    root = here.parent
elif in_colab:
    root = Path('/content/os-project-26-2')
    if not (root / 'protection').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/d86729/os-project-26-2.git', str(root)], check=True)
else:
    raise RuntimeError('저장소 루트나 presentation 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(root))
from protection.runner import initialize_single_thread
initialize_single_thread()
import numpy, scipy, networkx
print('저장소 준비 완료')

저장소 준비 완료


## 2. 시드로 문제 만들기

기지는 0~1 좌표에 놓인다. 처음 세 기지를 삼각형으로 잇고, 이후 각 기지를 기존 기지 두 곳에 연결한다. 연결 시 기존 차수와 거리를 고려한다. 선의 용량·비용과 통신 수요도 같은 난수 생성기로 정한다. 같은 생성 규칙과 호환되는 실행 환경에서 같은 시드를 쓰면 같은 문제가 생성된다.

공격 후보는 기지, 선의 중점, 격자점으로 만들고 각 위치에서 타격되는 선 목록을 계산해 입력에 넣는다. **가능한 공격들을 모두 알려 주며, 실제 최악 공격은 방호안에 따라 채점 때 결정한다.**

In [2]:
from protection.generation import generate_scenario
from protection.evaluation import budget_for
from protection.models import ScenarioBundle, save_bundle
from protection.program import scenario_input

n, seed, budget_percent = 6, 6, 20
scenario = generate_scenario(n, seed)
budget = budget_for(scenario, budget_percent)
bundle = ScenarioBundle(scenario, budget_percent, budget)
work = tempfile.TemporaryDirectory(prefix='protection-demo-')
workdir = Path(work.name)
save_bundle(bundle, workdir / 'problem.json')
(workdir / 'challenge.txt').write_bytes(scenario_input(bundle))
print(f'기지 {n}개, 통신선 {len(scenario.edges)}개, 예산 {budget}, 시드 {seed}')
print('problem.json과 challenge.txt 생성 완료')
print('입력 앞부분:')
print('\n'.join((workdir / 'challenge.txt').read_text().splitlines()[:5]))

기지 6개, 통신선 9개, 예산 8, 시드 6
problem.json과 challenge.txt 생성 완료
입력 앞부분:
6 9 6 51
8 20 6 -1
0.5381643514719432 0.34327086981333843
0.36906723979537825 0.37449676558788236
0.9874449901864666 0.6327562726071461


실제 문제 생성 함수는 아래에서 확인할 수 있다. 연결 확률 계산은 별도 보조 함수를 사용한다.

In [3]:
import inspect
from protection.generation import connection_probabilities
print(inspect.getsource(connection_probabilities))
print(inspect.getsource(generate_scenario))

def connection_probabilities(coordinates, degrees, new_id, candidates):
    distances = np.linalg.norm(coordinates[np.asarray(candidates)] - coordinates[new_id], axis=1)
    weights = np.asarray([degrees[i] for i in candidates], dtype=np.float64) / np.maximum(distances, 1e-9)**3
    total = weights.sum()
    require(np.isfinite(weights).all() and np.isfinite(total) and total > 0,
            'nonfinite or unnormalizable attachment weights')
    probabilities = weights / total
    require(np.isfinite(probabilities).all(), 'invalid attachment probabilities')
    return probabilities

def generate_scenario(n: int, seed: int) -> Scenario:
    integer(n, 'n', 3)
    integer(seed, 'seed')
    rng = np.random.default_rng(seed)
    coordinates = rng.random((n, 2))
    edges = {(0, 1), (0, 2), (1, 2)}
    degrees = [2, 2, 2] + [0] * (n - 3)
    # 삼각형으로 시작해 각 신규 노드를 서로 다른 기존 노드 두 곳에 연결한다.
    for new_id in range(3, n):
        candidates = list(range(new_id))
        for _ in range(2):
         

텍스트 입력 순서는 다음과 같다.

1. `기지 수  선 수  수요 수  공격 후보 수`
2. `예산  예산 비율  문제 시드  전략 시드` (전략 시드가 없으면 `-1`)
3. 기지 좌표, 선 정보, 수요 정보, 공격 후보 정보가 각각 개수만큼 이어진다.
4. 각 공격 후보에는 해당 위치에서 타격 가능한 선 번호가 들어 있다.

출력은 `선택 개수` 다음에 **오름차순 선 번호**를 쓴다. 예를 들어 `2 1 4`는 1번과 4번 선을 선택했다는 뜻이다.

## 3. 기준선 실행: STDIN과 STDOUT

기존 차수/비용 기준선 파일을 임시 폴더에 복사한다. 차수는 기지에 직접 연결된 선의 수다. 각 선에 대해 양 끝 기지의 차수 합을 방호 비용으로 나눈 뒤, 높은 순서로 예산 안에서 선택한다. 공격과 수요는 이 기준선의 선택에는 사용하지 않는다.

아래 코드는 실제 실험에 사용한 기존 예시다. 팀원별 최종 코드는 각자가 작성·검토하고 같은 입력·출력 계약을 따라야 한다.

In [4]:
import shutil
source = root / 'results/agent-study/2026-11-22/agent-1/degree.py'
shutil.copyfile(source, workdir / '기준선.py')
os.chdir(workdir)
print((workdir / '기준선.py').read_text())

"""AI research draft, agent 1: endpoint-degree-per-cost baseline.

One deterministic pass; ignore demands and attack coordinates deliberately.
Same selection rule as betweenness.py, so the ranking is the controlled variable.
No scenario-specific lookup or optimal-reference access.
"""
from fractions import Fraction
import sys


def read_problem(stream):
    n, m, d, a = map(int, stream.readline().split())
    budget = int(stream.readline().split()[0])
    for _ in range(n):
        x, y = map(float, stream.readline().split())
    edges = [tuple(map(int, stream.readline().split())) for _ in range(m)]
    for _ in range(d + a):
        stream.readline()
    return n, edges, budget


def select(n, edges, budget):
    degree = [0] * n
    for eid, u, v, capacity, cost in edges:
        degree[u] += 1
        degree[v] += 1
    order = sorted(edges, key=lambda e: (-Fraction(degree[e[1]] + degree[e[2]], e[4]), e[4], e[0]))
    chosen = []
    for eid, u, v, capacity, cost in order:
        i

In [5]:
!python3 기준선.py < challenge.txt > answer.txt
!cat answer.txt

4 0 5 7 8


위 명령에서 `< challenge.txt`는 파일 내용을 표준 입력으로 보내고, `> answer.txt`는 표준 출력을 파일에 저장한다. Colab 코드 셀에서는 `!`로 셸 명령을 실행한다. Linux 터미널에서는 `!` 없이 같은 명령을 쓴다.

`python3 김연우.py < challenge.txt > answer.txt`도 같은 방식이다. 단, 파일 이름은 실제 팀원이 직접 작성해 저장한 이름으로 바꾸어야 한다.

## 4. 방호안 점수와 최적 참조

공통 평가는 보호한 선을 유지하고 공격 후보별 피해를 적용한다. **연결된 기지 쌍 수가 가장 적은 공격**을 고르고, 동점이면 **전달률이 더 낮은 공격**을 고른다. 보호 전략은 그 최악 공격의 점수가 더 높아지도록 한다.

이 예제에서는 별도로 완전탐색을 마친 동일 문제의 저장된 참조를 불러온다. 최적 참조는 모든 문제에서 항상 있는 것이 아니며, 이 셀은 새로운 완전탐색을 실행하지 않는다.

In [6]:
import warnings
from scipy.optimize import OptimizeWarning
warnings.filterwarnings('ignore', message='Unrecognized options detected:.*', category=OptimizeWarning)
from protection.evaluation import Evaluator
from protection.models import load_bundle
from protection.program import protected_ids

answer = (workdir / 'answer.txt').read_bytes()
chosen = protected_ids(answer, bundle)
score = Evaluator(scenario).evaluate(list(chosen), budget)
print('선택한 선:', list(chosen))
print('사용 비용:', score.cost, '/', budget)
print('최악 공격 후 연결 쌍:', score.connected_pairs)
print('최악 공격 후 전달률:', f'{score.delivery_ratio:.2%}')

reference_path = root / 'scenarios/problem-bank/bundles' / f'n{n}-seed{seed}-p{budget_percent}.json'
if reference_path.is_file():
    reference_bundle = load_bundle(reference_path, require_reference=True)
    if bundle.case_id == reference_bundle.case_id:
        optimum = reference_bundle.optimal_reference.evaluation
        print('저장된 최적 연결 쌍:', optimum.connected_pairs)
        print('저장된 최적 전달률:', f'{optimum.delivery_ratio:.2%}')
        print('최적 점수 일치:', score.score == optimum.score)
    else:
        print('생성 결과가 저장된 참조와 달라 최적 점수를 비교하지 않습니다.')
else:
    print('이 시드와 예산에 대한 인증 참조 파일이 없습니다.')

선택한 선: [0, 5, 7, 8]
사용 비용: 7 / 8
최악 공격 후 연결 쌍: 10
최악 공격 후 전달률: 57.14%
저장된 최적 연결 쌍: 10
저장된 최적 전달률: 57.14%
최적 점수 일치: True


## 5. 팀원별 전략과 실험

- **1 김대건:** 차수/비용 기준선과 문제 생성 과정을 설명한다.
- **2 신동윤, 3 김연우:** 각자 직접 작성한 프로그램의 입력·선택 기준·출력을 설명한다. 완성 파일을 이 폴더에 넣고 위 명령의 파일명만 바꿔 실행한다.
- **4 진명호:** 공통 점수와, 예산 안의 방호안을 모두 조사하는 완전탐색 원리를 설명한다.

전략을 고친 뒤에는 그 코드를 동일한 문제들에서 다시 실행해야 한다. 이전 코드의 점수를 새 코드의 결과로 사용하지 않는다. 이 시연 셀은 자원 제한을 강제하지 않는다. 실험의 시간·메모리 한도와 실제 실행 환경은 결과 기록에서 확인한다.

## 면담에서 설명할 질문

1. 같은 시드를 쓰면 왜 문제를 다시 만들 수 있는가?
2. 입력으로 받은 공격 후보와 실제 최악 공격은 어떻게 다른가?
3. 자기 전략은 어떤 기준으로 선을 고르고, 예산을 어떻게 지키는가?
4. 연결 쌍과 전달률은 어떤 순서로 비교하는가?
5. 완전탐색은 무엇을 모두 살펴보며, 큰 문제에서 왜 어려운가?
6. 최적 참조가 없는 문제에서는 무엇을 비교할 수 있는가?

## 선택: 이전 20사례 기준선 재현

아래는 시연 후 필요할 때만 실행한다. n=8·12·16·24·32의 10개 그래프와 두 예산에서 차수와 매개중심성을 다시 실행하고 저장된 공식 점수를 확인한다. 8사례에만 인증 최적 참조가 있다. 두 예산은 같은 그래프에서 나왔으며 20개 독립 그래프가 아니다.

**실행하려면** 아래 첫 줄을 `True`로 바꾼다. CPU 한 코어·실행당 300초·프로세스 주소 공간 6GB의 로컬 한도를 적용하며 공통 채점 시간은 별도다. 웹의 cgroup 한도와는 다르다.

In [7]:
RUN_FULL_STUDY = False  # 20사례 재현 시 True로 변경
if RUN_FULL_STUDY:
    os.chdir(root)
    from pathlib import Path
    import hashlib
    import json
    import os
    import resource
    import subprocess
    import sys

    from protection.runner import initialize_single_thread, collect_environment
    initialize_single_thread()
    from protection.models import load_bundle
    from protection.program import protected_ids, scenario_input
    from protection.evaluation import Evaluator

    root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'pyproject.toml').is_file())
    study = root / 'results/agent-study/2026-11-24-sized'
    protocol = json.loads((study / 'protocol.json').read_text(encoding='utf-8'))
    cases = json.loads((study / 'cases.json').read_text(encoding='utf-8'))
    saved = [json.loads(line) for line in (study / 'runs.jsonl').read_text(encoding='utf-8').splitlines()]
    methods = {item['method']: item for item in protocol['strategies']}
    assert len(cases) == 20
    environment = collect_environment()
    print('운영체제:', environment['os'])
    print('CPU:', environment['cpu'])
    print('Python:', environment['python'])
    print('사례:', len(cases), '· 그래프:', len({(c['n'], c['seed']) for c in cases}))
    from collections import Counter
    from IPython.display import Markdown, display

    def limit_child():
        resource.setrlimit(resource.RLIMIT_AS, (6_000_000_000, 6_000_000_000))
        available = os.sched_getaffinity(0)
        os.sched_setaffinity(0, {min(available)})

    outcomes = Counter()
    lines = ['| 사례 | 매개중심성 연결 쌍 / 전달 점수 | 차수 연결 쌍 / 전달 점수 | 결과 | 최적 참조 |',
             '|---|---:|---:|---|---|']
    for case in cases:
        bundle = load_bundle(root / case['path'], require_reference=case['reference_available'])
        assert bundle.case_id == case['case_id']
        stdin = scenario_input(bundle)
        assert hashlib.sha256(stdin).hexdigest() == case['stdin_sha256']
        scores = {}
        for name in ('degree', 'betweenness'):
            method = methods[name]
            source = root / method['path']
            assert hashlib.sha256(source.read_bytes()).hexdigest() == method['sha256']
            process = subprocess.run([sys.executable, '-I', str(source)], input=stdin,
                                     capture_output=True, timeout=300, check=True,
                                     preexec_fn=limit_child)
            chosen = protected_ids(process.stdout, bundle)
            score = Evaluator(bundle.scenario).evaluate(list(chosen), bundle.budget)
            previous = next(r for r in saved if r['name'] == case['name'] and r['method'] == name)
            assert previous['status'] == 'completed'
            assert (score.connected_pairs, score.delivery_units) == (previous['connected_pairs'],
                                                                      previous['delivery_units'])
            scores[name] = score
        a, b = scores['betweenness'], scores['degree']
        outcome = '매개중심성' if a.score > b.score else '차수' if a.score < b.score else '동점'
        outcomes[outcome] += 1
        lines.append(f"| {case['name']} | {a.connected_pairs} / {a.delivery_units} | "
                     f"{b.connected_pairs} / {b.delivery_units} | {outcome} | "
                     f"{'있음' if case['reference_available'] else '미확인'} |")
    display(Markdown(chr(10).join(lines)))
    print('매개중심성 승:', outcomes['매개중심성'], '동점:', outcomes['동점'],
          '차수 승:', outcomes['차수'])
else:
    print('20사례 재현을 건너뜁니다. 실행하려면 RUN_FULL_STUDY = True로 바꾸세요.')

20사례 재현을 건너뜁니다. 실행하려면 RUN_FULL_STUDY = True로 바꾸세요.
